In [ ]:
# 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Note on Dataset, Data. Refer to the archived source for the detailed original comment.
!ls /content/drive/MyDrive/datasets/DIV2K_HRLR/

benchmark	DIV2K_train_LR_bicubic	DIV2K_valid_LR_bicubic
DIV2K_train_HR	DIV2K_valid_HR		requirements.txt


### Experiment Notes

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [49]:
# Note on Define, Path. Refer to the archived source for the detailed original comment.
def get_hr_folder(split_info, scale):
    hr_base = split_info['hr_folder']
    hr_cropped_dir = os.path.join(hr_base, 'HR_cropped', f'X{scale}')
    if os.path.isdir(hr_cropped_dir):
        return hr_cropped_dir
    else:
        return hr_base

# Note on Dataset, Compute, Data. Refer to the archived source for the detailed original comment.
def evaluate_model(model, loader, psnr_model, ssim_model, device):
    model.eval()
    psnr_vals, ssim_vals = [], []
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            sr = model(lr)

            # Note on Image. Refer to the archived source for the detailed original comment.
            if sr.shape != hr.shape:
                raise ValueError(
                    f"SR/HR size mismatch: "
                    f"SR={tuple(sr.shape)}, "
                    f"HR={tuple(hr.shape)}"
                )
            psnr_vals.append(psnr_model(sr, hr).item())
            ssim_vals.append(ssim_model(sr, hr).item())
    return np.mean(psnr_vals), np.mean(ssim_vals)

# Note on Define, Path. Refer to the archived source for the detailed original comment.
splits_info_multi = {
    'train': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X4'
        }
    },
    'valid': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X4'
        }
    },
    'Set5': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X4'
        }
    },
    'Set14': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X4'
        }
    },
    'Urban100': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X4'
        }
    },
    'BSDS100': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X4'
        }
    },
}

### Image / Save / Question

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.

×2 and ×3 LR images were generated from HR images using bicubic downsampling.


In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def create_lr_images(
    hr_dir,
    save_root,
    scales=[2, 3, 4],
    ext='.png',
    crop_mode='center',
    save_hr=False
):
    # Note on Save. Refer to the archived source for the detailed original comment.
    if save_hr:
        hr_save_root = os.path.join(hr_dir, 'HR_cropped')
    else:
        hr_save_root = None

    for scale in scales:
        lr_save_dir = os.path.join(save_root, f'X{scale}')
        os.makedirs(lr_save_dir, exist_ok=True)

        if save_hr:
            hr_save_dir = os.path.join(hr_save_root, f'X{scale}')
            os.makedirs(hr_save_dir, exist_ok=True)

        hr_files = [f for f in os.listdir(hr_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]

        for fname in tqdm(hr_files, desc=f"Generating LR x{scale}"):
            hr_path = os.path.join(hr_dir, fname)
            hr_img = Image.open(hr_path).convert('RGB')
            w, h = hr_img.size

            if w % scale != 0 or h % scale != 0:
                new_w = w - (w % scale)
                new_h = h - (h % scale)

                if crop_mode == 'center':
                    left = (w - new_w) // 2
                    top  = (h - new_h) // 2
                elif crop_mode == 'random':
                    import random
                    left = random.randint(0, w - new_w)
                    top  = random.randint(0, h - new_h)
                else:
                    raise ValueError("crop_mode must be 'center' or 'random'")

                hr_cropped = hr_img.crop((left, top, left + new_w, top + new_h))
            else:
                hr_cropped = hr_img
                new_w, new_h = w, h

            lr_size = (new_w // scale, new_h // scale)
            lr_img  = hr_cropped.resize(lr_size, Image.BICUBIC)

            base, _ = os.path.splitext(fname)
            lr_fname = f"{base}x{scale}{ext}"
            lr_img.save(os.path.join(lr_save_dir, lr_fname))

            if save_hr:
                hr_cropped.save(os.path.join(hr_save_dir, fname))   # 保持原名

    print("\nDone! LR images and cropped HR (if enabled) are ready.\n")

In [ ]:
# Note on Training, Generate. Refer to the archived source for the detailed original comment.
create_lr_images(
    hr_dir="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR",
    save_root="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic",
    scales=[2,3]
)

In [ ]:
# Note on Validation, Generate. Refer to the archived source for the detailed original comment.
create_lr_images(
    hr_dir="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_HR",
    save_root="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic",
    scales=[2,3]
)

Generating LR x3: 100%|██████████| 100/100 [00:14<00:00,  7.04it/s]


Done!


In [ ]:
# Note on Generate. Refer to the archived source for the detailed original comment.
# Set5
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic',
    scales=[2,3]
)

# Set14
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic',
    scales=[2,3]
)

# Urban100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic',
    scales=[2,3]
)

# BSDS100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic',
    scales=[2,3]
)

Generating LR x3: 100%|██████████| 5/5 [00:00<00:00, 127.64it/s]



Done!


Generating LR x3: 100%|██████████| 14/14 [00:00<00:00, 81.34it/s]



Done!


Generating LR x3: 100%|██████████| 100/100 [00:04<00:00, 24.79it/s]



Done!


Generating LR x3: 100%|██████████| 100/100 [00:00<00:00, 100.74it/s]


Done!


### Question / Generate

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
# Note on Training, Generate. Refer to the archived source for the detailed original comment.
# Note on Training, Data. Refer to the archived source for the detailed original comment.
create_lr_images(
    hr_dir="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR",
    save_root="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic",
    scales=[3],
    crop_mode='random',
    save_hr=True
)

Generating LR x3: 100%|██████████| 800/800 [21:32<00:00,  1.62s/it]


Done! LR images and cropped HR (if enabled) are ready.



In [ ]:
# Note on Validation, Generate. Refer to the archived source for the detailed original comment.
# Note on Validation. Refer to the archived source for the detailed original comment.
create_lr_images(
    hr_dir="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_HR",
    save_root="/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic",
    scales=[3],
    crop_mode='center',
    save_hr=True
)

Generating LR x3: 100%|██████████| 100/100 [01:20<00:00,  1.24it/s]


Done! LR images and cropped HR (if enabled) are ready.


In [ ]:
# Note on Generate. Refer to the archived source for the detailed original comment.
# Set5
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic',
    scales=[3],
    crop_mode='center',
    save_hr=True
)

# Set14
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic',
    scales=[3],
    crop_mode='center',
    save_hr=True
)

# Urban100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic',
    scales=[3],
    crop_mode='center',
    save_hr=True
)

# BSDS100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic',
    scales=[3],
    crop_mode='center',
    save_hr=True
)

Generating LR x3: 100%|██████████| 5/5 [00:03<00:00,  1.42it/s]



Done! LR images and cropped HR (if enabled) are ready.


Generating LR x3: 100%|██████████| 14/14 [00:09<00:00,  1.48it/s]



Done! LR images and cropped HR (if enabled) are ready.


Generating LR x3: 100%|██████████| 100/100 [01:35<00:00,  1.05it/s]



Done! LR images and cropped HR (if enabled) are ready.


Generating LR x3: 100%|██████████| 100/100 [01:07<00:00,  1.49it/s]


Done! LR images and cropped HR (if enabled) are ready.


In [37]:
# Note on Generate. Refer to the archived source for the detailed original comment.
# Set5
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic',
    scales=[2],
    crop_mode='center',
    save_hr=True
)

# Set14
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic',
    scales=[2],
    crop_mode='center',
    save_hr=True
)

# Urban100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic',
    scales=[2],
    crop_mode='center',
    save_hr=True
)

# BSDS100
create_lr_images(
    hr_dir='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/HR',
    save_root='/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic',
    scales=[2],
    crop_mode='center',
    save_hr=True
)

Generating LR x2: 100%|██████████| 5/5 [00:00<00:00, 25.86it/s]



Done! LR images and cropped HR (if enabled) are ready.



Generating LR x2: 100%|██████████| 14/14 [00:00<00:00, 14.85it/s]



Done! LR images and cropped HR (if enabled) are ready.



Generating LR x2: 100%|██████████| 100/100 [00:22<00:00,  4.51it/s]



Done! LR images and cropped HR (if enabled) are ready.



Generating LR x2: 100%|██████████| 100/100 [00:04<00:00, 20.13it/s]


Done! LR images and cropped HR (if enabled) are ready.



### Generate

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [38]:
import os
import csv

def generate_csv_scales(split, image_folder, lr_folder, scale, output_dir):
    """
    为指定的 split 生成 CSV 文件，保存到 output_dir/{split}_images_x{scale}.csv

    Args:
        split (str): 数据集划分标识，如 'train', 'valid', 'Set5' 等
        image_folder (str): HR 图像文件夹路径
        lr_folder (str): LR 图像文件夹路径（对应 scale）
        scale (str or int): LR 下采样倍数 xN
        output_dir (str): 输出目录（会自动创建）
    """
    os.makedirs(output_dir, exist_ok=True)

    output_csv = os.path.join(output_dir, f"{split}_images_x{scale}.csv")

    hr_files = sorted(os.listdir(image_folder))
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        for fname in hr_files:
            base, ext = os.path.splitext(fname)
            lr_fname = f"{base}x{scale}{ext}"
            lr_path = os.path.join(lr_folder, lr_fname)
            if os.path.exists(lr_path):
                writer.writerow([fname, lr_fname])
            else:
                print(f"Warning: {lr_fname} not found in LR folder for split {split}")

    print(f"CSV generated for split '{split}' (x{scale}) at {output_csv}")

In [39]:
output_dir = "/content/drive/MyDrive/DIV2K_Results/output_csv"

scales = ['2', '3', '4']  # 可以选 2、3 或全部

for split, paths in splits_info_multi.items():
    hr_base = paths["hr_folder"]   # 原始 HR 根目录
    for scale in scales:
        # Note on Use. Refer to the archived source for the detailed original comment.
        hr_cropped_dir = os.path.join(hr_base, 'HR_cropped', f'X{scale}')
        if os.path.isdir(hr_cropped_dir):
            hr_folder_to_use = hr_cropped_dir
        else:
            hr_folder_to_use = hr_base   # 没有裁剪版本时，退回原始目录

        generate_csv_scales(
            split=split,
            image_folder=hr_folder_to_use,
            lr_folder=paths["lr_folders"][scale],
            scale=scale,
            output_dir=output_dir
        )

CSV generated for split 'train' (x2) at /content/drive/MyDrive/DIV2K_Results/output_csv/train_images_x2.csv
CSV generated for split 'train' (x3) at /content/drive/MyDrive/DIV2K_Results/output_csv/train_images_x3.csv
CSV generated for split 'train' (x4) at /content/drive/MyDrive/DIV2K_Results/output_csv/train_images_x4.csv
CSV generated for split 'valid' (x2) at /content/drive/MyDrive/DIV2K_Results/output_csv/valid_images_x2.csv
CSV generated for split 'valid' (x3) at /content/drive/MyDrive/DIV2K_Results/output_csv/valid_images_x3.csv
CSV generated for split 'valid' (x4) at /content/drive/MyDrive/DIV2K_Results/output_csv/valid_images_x4.csv
CSV generated for split 'Set5' (x2) at /content/drive/MyDrive/DIV2K_Results/output_csv/Set5_images_x2.csv
CSV generated for split 'Set5' (x3) at /content/drive/MyDrive/DIV2K_Results/output_csv/Set5_images_x3.csv
CSV generated for split 'Set5' (x4) at /content/drive/MyDrive/DIV2K_Results/output_csv/Set5_images_x4.csv
CSV generated for split 'Set14' (x

### Experiment Notes

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
!pip install -r /content/drive/MyDrive/datasets/DIV2K_HRLR/requirements.txt

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


### Code

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
!git clone https://github.com/Lornatang/SRGAN-PyTorch

Cloning into 'SRGAN-PyTorch'...
remote: Enumerating objects: 3634, done.
remote: Counting objects: 100% (923/923), done.
remote: Compressing objects: 100% (286/286), done.
remote: Total 3634 (delta 690), reused 810 (delta 633), pack-reused 2711 (from 1)
Receiving objects: 100% (3634/3634), 2.98 MiB | 24.58 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.


In [ ]:
# Note on Path. Refer to the archived source for the detailed original comment.

import sys
sys.path.append('/content/SRGAN-PyTorch')

### Training

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
%%writefile sr_dataset_exp_scale.py

import csv
import os
import random
from dataclasses import dataclass

import torchvision.transforms.functional as TF
import torch
from torch import Tensor
from torch.utils.data import Dataset
from torchvision.io import decode_image
from tqdm import tqdm


@dataclass
class SRDatasetConfig:
    csv_folder: str = "/content/drive/MyDrive/DIV2K_Results/output_csv"
    HR_data_folder: str = '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR'
    LR_data_folder: str = '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X4'
    crop_size: int = 96
    scaling_factor: int = 4
    patch_num: int = 100
    cache: bool = True
    scale: int = 4
    color_augment: bool = False
    noise_augment: bool = False  # 是否开启高级增强


class SRDataset(Dataset):
    def __init__(self, split: str, config: SRDatasetConfig):
        self.cache = config.cache
        self.split = split
        self.csv_folder = config.csv_folder
        self.HR_data_folder = config.HR_data_folder
        self.LR_data_folder = config.LR_data_folder
        self.crop_size = int(config.crop_size)
        self.scaling_factor = int(config.scaling_factor)
        self.patch_num = config.patch_num
        self.scale = config.scale
        self.color_augment = config.color_augment
        self.noise_augment = config.noise_augment

        assert self.split in {"train", "valid", "Set5", "Set14", "B100", "Urban100", "BSDS100"}

        self.HR_images = []
        self.LR_images = []
        csv_name = f"{split}_images_x{self.scale}.csv"
        with open(os.path.join(self.csv_folder, csv_name), "r") as f:
            reader = csv.reader(f)
            for line in reader:
                self.HR_images.append(line[0])
                if os.path.exists(os.path.join(self.LR_data_folder, line[1])):
                    self.LR_images.append(line[1])
                else:
                    assert os.path.exists(os.path.join(self.LR_data_folder, line[0]))
                    self.LR_images.append(line[0])
        if self.cache:
            self.images_cache = []
            for i in tqdm(range(len(self.HR_images)), desc="Loading dataset"):
                # Load HR and LR images using torchvision
                hr_path = os.path.join(self.HR_data_folder, self.HR_images[i])
                lr_path = os.path.join(self.LR_data_folder, self.LR_images[i])

                hr_img = (
                    decode_image(hr_path, mode="RGB").float() / 255.0
                )  # C x H x W, float32
                lr_img = decode_image(lr_path, mode="RGB").float() / 255.0

                # Ensure size consistency
                hr_img, lr_img = self._center_crop_and_resize_if_needed(hr_img, lr_img)
                self.images_cache.append((hr_img, lr_img))

    def _center_crop_and_resize_if_needed(self, hr_img: Tensor, lr_img: Tensor):
        # Only if HR and LR shapes are inconsistent
        C, hr_h, hr_w = hr_img.shape
        C2, lr_h, lr_w = lr_img.shape
        assert C == C2

        expected_lr_h = hr_h // self.scaling_factor
        expected_lr_w = hr_w // self.scaling_factor

        if lr_h != expected_lr_h or lr_w != expected_lr_w:
            # Center crop HR to be divisible by scaling factor
            new_hr_h = (hr_h // self.scaling_factor) * self.scaling_factor
            new_hr_w = (hr_w // self.scaling_factor) * self.scaling_factor
            top = (hr_h - new_hr_h) // 2
            left = (hr_w - new_hr_w) // 2
            hr_img = TF.crop(hr_img, top, left, new_hr_h, new_hr_w)

            # Resize LR image from HR
            lr_img = TF.resize(
                hr_img,
                [new_hr_h // self.scaling_factor, new_hr_w // self.scaling_factor],
                interpolation=TF.InterpolationMode.BICUBIC,
            )
        return hr_img, lr_img

    def __getitem__(self, i: int):
        if self.split == "train":
            i = i // self.patch_num

        if self.cache:
            hr_img, lr_img = self.images_cache[i]
        else:
            # Load HR and LR images using torchvision
            hr_path = os.path.join(self.HR_data_folder, self.HR_images[i])
            lr_path = os.path.join(self.LR_data_folder, self.LR_images[i])

            hr_img = (
                decode_image(hr_path, mode="RGB").float() / 255.0
            )  # C x H x W, float32
            lr_img = decode_image(lr_path, mode="RGB").float() / 255.0

            # Ensure size consistency
            hr_img, lr_img = self._center_crop_and_resize_if_needed(hr_img, lr_img)

        if self.split == "train":
            lr_img, hr_img = self._sample_patch(lr_img, hr_img)
            lr_img, hr_img = self._augment(lr_img, hr_img)

        return lr_img, hr_img

    def _sample_patch(self, lr_img: Tensor, hr_img: Tensor):
        _, h, w = lr_img.shape
        lr_crop_size = self.crop_size // self.scaling_factor
        left = random.randint(2, w - lr_crop_size - 2)
        top = random.randint(2, h - lr_crop_size - 2)

        lr_patch = TF.crop(lr_img, top, left, lr_crop_size, lr_crop_size)
        hr_patch = TF.crop(
            hr_img,
            top * self.scaling_factor,
            left * self.scaling_factor,
            self.crop_size,
            self.crop_size,
        )
        return lr_patch, hr_patch

    def _augment(self, lr_img: Tensor, hr_img: Tensor):
        if random.random() < 0.5:
            lr_img = TF.hflip(lr_img)
            hr_img = TF.hflip(hr_img)
        if random.random() < 0.5:
            lr_img = TF.vflip(lr_img)
            hr_img = TF.vflip(hr_img)
        if random.random() < 0.5:
            angle = 90
            lr_img = TF.rotate(lr_img, angle)
            hr_img = TF.rotate(hr_img, angle)

        # 新增颜色抖动（亮度、对比度、饱和度）
        # 颜色增强（仅当 color_augment=True 时启用）
        if self.color_augment:
            if random.random() < 0.5:
                brightness_factor = random.uniform(0.8, 1.2)
                contrast_factor = random.uniform(0.8, 1.2)
                saturation_factor = random.uniform(0.8, 1.2)
                lr_img = TF.adjust_brightness(lr_img, brightness_factor)
                hr_img = TF.adjust_brightness(hr_img, brightness_factor)
                lr_img = TF.adjust_contrast(lr_img, contrast_factor)
                hr_img = TF.adjust_contrast(hr_img, contrast_factor)
                lr_img = TF.adjust_saturation(lr_img, saturation_factor)
                hr_img = TF.adjust_saturation(hr_img, saturation_factor)

        if self.noise_augment:
            # 高斯噪声（仅加在 LR 上）
            if random.random() < 0.3:
                noise = torch.randn_like(lr_img) * 0.02
                lr_img = lr_img + noise
                lr_img = torch.clamp(lr_img, 0.0, 1.0)

        return lr_img, hr_img

    def __len__(self):

        if self.split == "train":
            return len(self.HR_images) * self.patch_num
        else:
            return len(self.HR_images)

Writing sr_dataset_exp_scale.py


In [ ]:
%%writefile train_srresnet_exp_scale.py

import os
import time
import argparse
import random
import numpy as np
import csv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import MultiStepLR
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

import sys
sys.path.append('/content/SRGAN-PyTorch')

from sr_dataset_exp_scale import SRDatasetConfig, SRDataset
from model_srgan import SRResNet
from utils import AverageMeter, ProgressMeter
from image_quality_assessment import PSNR, SSIM

# ============= 配置 =============
# 随机种子
seed = 0

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_hr_folder(split_info, scale):
    hr_base = split_info['hr_folder']
    hr_cropped_dir = os.path.join(hr_base, 'HR_cropped', f'X{scale}')
    if os.path.isdir(hr_cropped_dir):
        return hr_cropped_dir
    else:
        return hr_base

splits_info_multi = {
    'train': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X4'
        }
    },
    'valid': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X4'
        }
    },
    'Set5': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X4'
        }
    },
    'Set14': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set14/LR_bicubic/X4'
        }
    },
    'Urban100': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Urban100/LR_bicubic/X4'
        }
    },
    'BSDS100': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/B100/LR_bicubic/X4'
        }
    },
}

def evaluate(eval_model, valid_loader, psnr_model, ssim_model, device):
    eval_model.eval()
    psnr_vals, ssim_vals = [], []
    with torch.no_grad():
        for lr_img, hr_img in valid_loader:
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            sr = eval_model(lr_img)
            psnr = psnr_model(sr, hr_img)
            ssim = ssim_model(sr, hr_img)
            psnr_vals.append(psnr.item())
            ssim_vals.append(ssim.item())
    avg_psnr = sum(psnr_vals) / len(psnr_vals)
    avg_ssim = sum(ssim_vals) / len(ssim_vals)
    return avg_psnr, avg_ssim

def main():
    parser = argparse.ArgumentParser(description='Train SRResNet')
    parser.add_argument('--lr', type=float, default=1e-4, help='Learning rate')
    parser.add_argument('--weight_decay', type=float, default=0.0, help='Weight decay')
    parser.add_argument('--channels', type=int, default=64, help='Number of channels in model')
    parser.add_argument('--epochs', type=int, default=100, help='Total epochs')
    parser.add_argument('--batch_size', type=int, default=16, help='Batch size')
    parser.add_argument('--crop_size', type=int, default=96, help='Crop size for training')
    parser.add_argument('--patch_num', type=int, default=100, help='Patches per image per epoch')
    parser.add_argument('--milestones', type=int, nargs='+', default=[40, 70, 90], help='LR decay milestones')
    parser.add_argument('--gamma', type=float, default=0.5, help='LR decay factor')
    parser.add_argument('--patience', type=int, default=20, help='Early stopping patience')
    parser.add_argument('--save_dir', type=str, default='/content/drive/MyDrive/DIV2K_Results', help='Directory to save models')
    parser.add_argument('--csv_folder', type=str, default='/content/drive/MyDrive/DIV2K_Results/output_csv', help='CSV folder')
    parser.add_argument('--run_name', type=str, default='srresnet_x4_exp', help='Name of this experiment')
    parser.add_argument('--num_workers', type=int, default=2, help='Number of data loading workers')
    parser.add_argument('--cache', action='store_true', help='Enable dataset caching')
    parser.add_argument('--scale', type=int, default=4, choices=[2,3,4], help='Upscaling factor')
    parser.add_argument('--color_augment', action='store_true', help='Enable color augmentation')
    parser.add_argument('--noise_augment', action='store_true', help='Add Gaussian noise to LR')
    args = parser.parse_args()

    # 随机种子
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    g_train = torch.Generator()
    g_train.manual_seed(seed)

    g_valid = torch.Generator()
    g_valid.manual_seed(seed + 1)

    # Note on Parameters. Refer to the archived source for the detailed original comment.
    lr = args.lr
    weight_decay = args.weight_decay
    channels = args.channels
    epochs = args.epochs
    batch_size = args.batch_size
    crop_size = args.crop_size
    patch_num = args.patch_num
    milestones = args.milestones
    gamma = args.gamma
    patience = args.patience
    save_dir = args.save_dir
    run_name = args.run_name
    os.makedirs(save_dir, exist_ok=True)
    csv_folder = args.csv_folder
    num_workers = args.num_workers
    cache = args.cache
    color_augment = args.color_augment
    noise_augment = args.noise_augment
    scale = args.scale

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Note on Path. Refer to the archived source for the detailed original comment.
    train_hr_folder = get_hr_folder(splits_info_multi['train'], scale)
    train_lr_folder = splits_info_multi['train']['lr_folders'][str(scale)]
    valid_hr_folder = get_hr_folder(splits_info_multi['valid'], scale)
    valid_lr_folder = splits_info_multi['valid']['lr_folders'][str(scale)]

    print(f"Train HR dir: {train_hr_folder}")
    print(f"Train LR dir: {train_lr_folder}")
    print(f"Valid HR dir: {valid_hr_folder}")
    print(f"Valid LR dir: {valid_lr_folder}")

    # Note on Training, Validation, Create. Refer to the archived source for the detailed original comment.
    train_config = SRDatasetConfig(
        csv_folder=csv_folder,
        HR_data_folder=train_hr_folder,
        LR_data_folder=train_lr_folder,
        crop_size=crop_size,
        scaling_factor=scale,
        patch_num=patch_num,
        cache=cache,
        scale=scale,
        color_augment=color_augment,
        noise_augment=noise_augment
    )
    valid_config = SRDatasetConfig(
        csv_folder=csv_folder,
        HR_data_folder=valid_hr_folder,
        LR_data_folder=valid_lr_folder,
        crop_size=-1,         # 验证时不裁剪
        scaling_factor=scale,
        patch_num=1,                # 验证时每个图像只取一次
        cache=cache,
        scale=scale,
        color_augment=False,
        noise_augment=False         # 验证集不高级增强
    )

    # Note on Dataset, Data. Refer to the archived source for the detailed original comment.
    train_dataset = SRDataset("train", train_config)
    valid_dataset = SRDataset("valid", valid_config)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        worker_init_fn=seed_worker,
        generator=g_train
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=1,                # 验证时 batch_size=1 方便计算每张图的指标
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g_valid
    )

    # Note on Model, Optimizer. Refer to the archived source for the detailed original comment.
    model = SRResNet(channels=channels, upscale=scale).to(device)

    ema_decay = 0.999
    ema_model = AveragedModel(
        model,
        multi_avg_fn=get_ema_multi_avg_fn(ema_decay),
        use_buffers=True
    )

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = MultiStepLR(optimizer, milestones=milestones, gamma=gamma)
    scaler = GradScaler()

    # Note on Validation, Evaluation. Refer to the archived source for the detailed original comment.
    psnr_model = PSNR(crop_border=scale, only_test_y_channel=True).to(device)
    ssim_model = SSIM(crop_border=scale, only_test_y_channel=True).to(device)

    # 最佳指标
    best_raw_psnr = 0.0
    best_ema_psnr = 0.0
    raw_save_path = os.path.join(save_dir, f'{run_name}_x{scale}_best_raw.pth')
    ema_save_path = os.path.join(save_dir, f'{run_name}_x{scale}_best_ema.pth')
    # 记录csv
    log_path = os.path.join(save_dir, f'{run_name}_x{scale}_metrics.csv')
    with open(log_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                'epoch',
                'raw_psnr',
                'raw_ssim',
                'ema_psnr',
                'ema_ssim',
                'lr'
            ])

    # 早停指标
    counter = 0

    # Note on Training, Loop. Refer to the archived source for the detailed original comment.
    for epoch in range(epochs):
        # Note on Training. Refer to the archived source for the detailed original comment.
        model.train()
        batch_time = AverageMeter('Time', ':6.3f')
        data_time = AverageMeter('Data', ':6.3f')
        losses = AverageMeter('Loss', ':6.6f')
        progress = ProgressMeter(len(train_loader), [batch_time, data_time, losses],
                                 prefix=f"Epoch [{epoch+1}/{epochs}]")

        end = time.time()
        for i, (lr_img, hr_img) in enumerate(train_loader):
            lr_img = lr_img.to(device, non_blocking=True)
            hr_img = hr_img.to(device, non_blocking=True)

            data_time.update(time.time() - end)

            optimizer.zero_grad()
            with autocast():
                sr = model(lr_img)
                loss = criterion(sr, hr_img)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # 更新 EMA
            ema_model.update_parameters(model)

            losses.update(loss.item(), lr_img.size(0))
            batch_time.update(time.time() - end)
            end = time.time()

            if i % 100 == 0:
                progress.display(i)

        # Note on Validation. Refer to the archived source for the detailed original comment.
        raw_psnr, raw_ssim = evaluate(model, valid_loader, psnr_model, ssim_model, device)
        ema_psnr, ema_ssim = evaluate(ema_model, valid_loader, psnr_model, ssim_model, device)
        print(f"[Epoch {epoch+1}] RAW: PSNR={raw_psnr:.2f}, SSIM={raw_ssim:.4f} | EMA: PSNR={ema_psnr:.2f}, SSIM={ema_ssim:.4f}")

        # Note on Validation. Refer to the archived source for the detailed original comment.
        with open(log_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch + 1,
                raw_psnr,
                raw_ssim,
                ema_psnr,
                ema_ssim,
                optimizer.param_groups[0]['lr']
            ])

        # Note on Save. Refer to the archived source for the detailed original comment.
        raw_improved = raw_psnr > best_raw_psnr
        ema_improved = ema_psnr > best_ema_psnr

        if raw_improved:
            best_raw_psnr = raw_psnr
            torch.save(model.state_dict(), raw_save_path)
            print(f"  -> Best RAW model saved (PSNR: {best_raw_psnr:.2f})")

        if ema_improved:
            best_ema_psnr = ema_psnr
            torch.save(ema_model.module.state_dict(), ema_save_path)
            print(f"  -> Best EMA model saved (PSNR: {best_ema_psnr:.2f})")

        if raw_improved or ema_improved:
            counter = 0
        else:
            counter += 1
            print(f"  -> No improvement for {counter} epochs.")

        if counter >= patience:
            print(
                f"Early stopping triggered at epoch {epoch+1}. "
                f"Best RAW PSNR: {best_raw_psnr:.2f}, "
                f"Best EMA PSNR: {best_ema_psnr:.2f}"
            )
            break

        scheduler.step()

    print("Training finished!")

if __name__ == "__main__":
    main()

Overwriting train_srresnet_exp_scale.py


### Training

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
# Note on Training. Refer to the archived source for the detailed original comment.

!python train_srresnet_exp_scale.py \
    --run_name "lr5e5_wd1e5_x2" \
    --lr 5e-5 \
    --weight_decay 1e-5 \
    --channels 64 \
    --epochs 100 \
    --scale 2 \
    --patience 20 \
    --cache

流式输出内容被截断，只能显示最后 5000 行内容。
Epoch [5/100][ 700/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.001)	Loss 0.001075 (0.001033)
Epoch [5/100][ 800/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000892 (0.001025)
Epoch [5/100][ 900/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000952 (0.001035)
Epoch [5/100][1000/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000832 (0.001034)
Epoch [5/100][1100/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.001695 (0.001034)
Epoch [5/100][1200/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000863 (0.001032)
Epoch [5/100][1300/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000744 (0.001031)
Epoch [5/100][1400/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000999 (0.001027)
Epoch [5/100][1500/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.000579 (0.001035)
Epoch [5/100][1600/5000]	Time  0.010 ( 0.010)	Data  0.000 ( 0.000)	Loss 0.001153 (0.001035)
Epoch [5/100][1700/5000]	Time  0.010 ( 0.010)	Data  0

### Results / Testing

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
%%writefile test_srresnet_exp_scale.py

import os
import sys
sys.path.append('/content/SRGAN-PyTorch')

import argparse
import torch
from torch.utils.data import DataLoader
from sr_dataset_exp_scale import SRDatasetConfig, SRDataset
from model_srgan import SRResNet
from image_quality_assessment import PSNR, SSIM
from utils import AverageMeter

def get_hr_folder(base_hr, scale):
    """
    如果 base_hr/HR_cropped/X{scale} 存在，则使用裁剪后的 HR，
    否则回退到原始 base_hr。
    """
    hr_cropped_dir = os.path.join(base_hr, 'HR_cropped', f'X{scale}')
    if os.path.isdir(hr_cropped_dir):
        return hr_cropped_dir
    return base_hr

def test_on_benchmark(model_path, benchmark_name, config_params, scale=4):
    """
    测试单个基准数据集
    Args:
        scale (int): 上采样倍数，2,3,4
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Note on Model. Refer to the archived source for the detailed original comment.
    model = SRResNet(channels=64, num_rcb=16, upscale=scale).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Note on Dataset, Data. Refer to the archived source for the detailed original comment.
    test_config = SRDatasetConfig(
        csv_folder=config_params['csv_folder'],
        HR_data_folder=config_params['image_folder'],
        LR_data_folder=config_params['lr_folder'],
        crop_size=-1,
        scaling_factor=scale,
        patch_num=1,
        cache=False,
        scale=scale,
        color_augment=False,
        noise_augment=False
    )
    test_dataset = SRDataset(benchmark_name, test_config)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

    # Note on Evaluation. Refer to the archived source for the detailed original comment.
    psnr_model = PSNR(crop_border=scale, only_test_y_channel=True).to(device)
    ssim_model = SSIM(crop_border=scale, only_test_y_channel=True).to(device)

    psnr_meter = AverageMeter('PSNR', ':6.2f')
    ssim_meter = AverageMeter('SSIM', ':6.4f')

    with torch.no_grad():
        for lr_img, hr_img in test_loader:
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            sr = model(lr_img)

            # Note on Image. Refer to the archived source for the detailed original comment.
            if sr.shape != hr_img.shape:
                min_h = min(sr.shape[2], hr_img.shape[2])
                min_w = min(sr.shape[3], hr_img.shape[3])
                sr = sr[:, :, :min_h, :min_w]
                hr_img = hr_img[:, :, :min_h, :min_w]

            psnr = psnr_model(sr, hr_img)
            ssim = ssim_model(sr, hr_img)
            psnr_meter.update(psnr.item())
            ssim_meter.update(ssim.item())

    print(f"{benchmark_name}: PSNR = {psnr_meter.avg:.2f} dB, SSIM = {ssim_meter.avg:.4f}")
    return psnr_meter.avg, ssim_meter.avg


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Test SRResNet on benchmarks")
    parser.add_argument('--model_path', type=str,
                        default="/content/drive/MyDrive/DIV2K_Results/srresnet_x4_best_raw.pth",
                        help="Path to the model checkpoint (.pth)")
    parser.add_argument('--scale', type=int, default=4, choices=[2,3,4],
                        help="Upscaling factor of the model")
    parser.add_argument('--csv_folder', type=str,
                        default="/content/drive/MyDrive/DIV2K_Results/output_csv",
                        help="Folder containing CSV files")
    args = parser.parse_args()

    scale = args.scale

    # Note on Dataset, Path, Data. Refer to the archived source for the detailed original comment.
    base_hr_root = "/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark"
    base_lr_root = "/content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark"

    benchmarks = {
        "Set5": {
            "image_folder": get_hr_folder(f"{base_hr_root}/Set5/HR", scale),
            "lr_folder": f"{base_lr_root}/Set5/LR_bicubic/X{scale}",
            "csv_folder": args.csv_folder
        },
        "Set14": {
            "image_folder": get_hr_folder(f"{base_hr_root}/Set14/HR", scale),
            "lr_folder": f"{base_lr_root}/Set14/LR_bicubic/X{scale}",
            "csv_folder": args.csv_folder
        },
        "Urban100": {
            "image_folder": get_hr_folder(f"{base_hr_root}/Urban100/HR", scale),
            "lr_folder": f"{base_lr_root}/Urban100/LR_bicubic/X{scale}",
            "csv_folder": args.csv_folder
        },
        "BSDS100": {
            "image_folder": get_hr_folder(f"{base_hr_root}/B100/HR", scale),
            "lr_folder": f"{base_lr_root}/B100/LR_bicubic/X{scale}",
            "csv_folder": args.csv_folder
        }
    }

    for name, paths in benchmarks.items():
        test_on_benchmark(
            model_path=args.model_path,
            benchmark_name=name,
            config_params=paths,
            scale=scale
        )

Overwriting test_srresnet_exp_scale.py


In [ ]:
# Note on Testing. Refer to the archived source for the detailed original comment.
model_paths = [
    "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x2_best_raw.pth",
    "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x2_best_ema.pth"
]
for path in model_paths:
    print(f"\n===== Testing {path} =====")
    !python test_srresnet_exp_scale.py --model_path {path} --scale 2


===== Testing /content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x2_best_raw.pth =====
Set5: PSNR = 37.73 dB, SSIM = 0.9584
Set14: PSNR = 32.08 dB, SSIM = 0.9032
Urban100: PSNR = 29.05 dB, SSIM = 0.8928
BSDS100: PSNR = 27.18 dB, SSIM = 0.8063

===== Testing /content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x2_best_ema.pth =====
Set5: PSNR = 37.80 dB, SSIM = 0.9587
Set14: PSNR = 32.13 dB, SSIM = 0.9039
Urban100: PSNR = 29.09 dB, SSIM = 0.8935
BSDS100: PSNR = 27.21 dB, SSIM = 0.8072


### Training

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


### Training / Parameters

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
%%writefile model_srgan.py

import math
import os
from typing import Any, cast, Dict, List, Union

import torch
from torch import nn, Tensor
from torch.nn import functional as F_torch
from torchvision import models, transforms
from torchvision.models.feature_extraction import create_feature_extractor

__all__ = [
    "DiscriminatorForVGG", "SRResNet",
    "discriminator_for_vgg", "srresnet_x2", "srresnet_x4", "srresnet_x8",
]

feature_extractor_net_cfgs: Dict[str, List[Union[str, int]]] = {
    "vgg11": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "vgg13": [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
    "vgg16": [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"],
    "vgg19": [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"],
}


def _make_layers(net_cfg_name: str, batch_norm: bool = False) -> nn.Sequential:
    net_cfg = feature_extractor_net_cfgs[net_cfg_name]
    layers: nn.Sequential[nn.Module] = nn.Sequential()
    in_channels = 3
    for v in net_cfg:
        if v == "M":
            layers.append(nn.MaxPool2d((2, 2), (2, 2)))
        else:
            v = cast(int, v)
            conv2d = nn.Conv2d(in_channels, v, (3, 3), (1, 1), (1, 1))
            if batch_norm:
                layers.append(conv2d)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(True))
            else:
                layers.append(conv2d)
                layers.append(nn.ReLU(True))
            in_channels = v

    return layers


class _FeatureExtractor(nn.Module):
    def __init__(
            self,
            net_cfg_name: str = "vgg19",
            batch_norm: bool = False,
            num_classes: int = 1000) -> None:
        super(_FeatureExtractor, self).__init__()
        self.features = _make_layers(net_cfg_name, batch_norm)

        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

        # Initialize neural network weights
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0, 0.01)
                nn.init.constant_(module.bias, 0)

    def forward(self, x: Tensor) -> Tensor:
        return self._forward_impl(x)

    # Support torch.script function
    def _forward_impl(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x


class SRResNet(nn.Module):
    def __init__(
            self,
            in_channels: int = 3,
            out_channels: int = 3,
            channels: int = 64,
            num_rcb: int = 16,
            upscale: int = 4,
    ) -> None:
        super(SRResNet, self).__init__()
        # Low frequency information extraction layer
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, channels, (9, 9), (1, 1), (4, 4)),
            nn.PReLU(),
        )

        # High frequency information extraction block
        trunk = []
        for _ in range(num_rcb):
            trunk.append(_ResidualConvBlock(channels))
        self.trunk = nn.Sequential(*trunk)

        # High-frequency information linear fusion layer
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(channels),
        )

        # zoom block
        upsampling = []
        if upscale in [2, 4, 8]:
            for _ in range(int(math.log(upscale, 2))):
                upsampling.append(
                    _UpsampleBlock(channels, 2)
                )

        elif upscale == 3:
            upsampling.append(
                _UpsampleBlock(channels, 3)
            )

        else:
            raise NotImplementedError(
                f"Upscale factor `{upscale}` is not supported."
            )

        self.upsampling = nn.Sequential(*upsampling)

        # reconstruction block
        self.conv3 = nn.Conv2d(channels, out_channels, (9, 9), (1, 1), (4, 4))

        # Initialize neural network weights
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)

    def forward(self, x: Tensor) -> Tensor:
        return self._forward_impl(x)

    # Support torch.script function
    def _forward_impl(self, x: Tensor) -> Tensor:
        conv1 = self.conv1(x)
        x = self.trunk(conv1)
        x = self.conv2(x)
        x = torch.add(x, conv1)
        x = self.upsampling(x)
        x = self.conv3(x)

        x = torch.clamp_(x, 0.0, 1.0)

        return x


class DiscriminatorForVGG(nn.Module):
    def __init__(
            self,
            in_channels: int = 3,
            out_channels: int = 1,
            channels: int = 64,
    ) -> None:
        super(DiscriminatorForVGG, self).__init__()
        self.features = nn.Sequential(
            # input size. (3) x 96 x 96
            nn.Conv2d(in_channels, channels, (3, 3), (1, 1), (1, 1), bias=True),
            nn.LeakyReLU(0.2, True),
            # state size. (64) x 48 x 48
            nn.Conv2d(channels, channels, (3, 3), (2, 2), (1, 1), bias=False),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(channels, int(2 * channels), (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(int(2 * channels)),
            nn.LeakyReLU(0.2, True),
            # state size. (128) x 24 x 24
            nn.Conv2d(int(2 * channels), int(2 * channels), (3, 3), (2, 2), (1, 1), bias=False),
            nn.BatchNorm2d(int(2 * channels)),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(int(2 * channels), int(4 * channels), (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(int(4 * channels)),
            nn.LeakyReLU(0.2, True),
            # state size. (256) x 12 x 12
            nn.Conv2d(int(4 * channels), int(4 * channels), (3, 3), (2, 2), (1, 1), bias=False),
            nn.BatchNorm2d(int(4 * channels)),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(int(4 * channels), int(8 * channels), (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(int(8 * channels)),
            nn.LeakyReLU(0.2, True),
            # state size. (512) x 6 x 6
            nn.Conv2d(int(8 * channels), int(8 * channels), (3, 3), (2, 2), (1, 1), bias=False),
            nn.BatchNorm2d(int(8 * channels)),
            nn.LeakyReLU(0.2, True),
        )

        self.classifier = nn.Sequential(
            nn.Linear(int(8 * channels) * 6 * 6, 1024),
            nn.LeakyReLU(0.2, True),
            nn.Linear(1024, out_channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        # Input image size must equal 96
        assert x.size(2) == 96 and x.size(3) == 96, "Input image size must be is 96x96"

        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x


class _ResidualConvBlock(nn.Module):
    def __init__(self, channels: int) -> None:
        super(_ResidualConvBlock, self).__init__()
        self.rcb = nn.Sequential(
            nn.Conv2d(channels, channels, (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, (3, 3), (1, 1), (1, 1), bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        x = self.rcb(x)

        x = torch.add(x, identity)

        return x


class _UpsampleBlock(nn.Module):
    def __init__(self, channels: int, upscale_factor: int) -> None:
        super(_UpsampleBlock, self).__init__()
        self.upsample_block = nn.Sequential(
            nn.Conv2d(channels, channels * upscale_factor * upscale_factor, (3, 3), (1, 1), (1, 1)),
            nn.PixelShuffle(upscale_factor),
            nn.PReLU(),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.upsample_block(x)

        return x


class ContentLoss(nn.Module):
    """Constructs a content loss function based on the VGG19 network.
    Using high-level feature mapping layers from the latter layers will focus more on the texture content of the image.

    Paper reference list:
        -`Photo-Realistic Single Image Super-Resolution Using a Generative Adversarial Network <https://arxiv.org/pdf/1609.04802.pdf>` paper.
        -`ESRGAN: Enhanced Super-Resolution Generative Adversarial Networks                    <https://arxiv.org/pdf/1809.00219.pdf>` paper.
        -`Perceptual Extreme Super Resolution Network with Receptive Field Block               <https://arxiv.org/pdf/2005.12597.pdf>` paper.

     """

    def __init__(
            self,
            net_cfg_name: str,
            batch_norm: bool,
            num_classes: int,
            model_weights_path: str,
            feature_nodes: list,
            feature_normalize_mean: list,
            feature_normalize_std: list,
    ) -> None:
        super(ContentLoss, self).__init__()
        # Define the feature extraction model
        model = _FeatureExtractor(net_cfg_name, batch_norm, num_classes)
        # Load the pre-trained model
        if model_weights_path == "":
            model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        elif model_weights_path is not None and os.path.exists(model_weights_path):
            checkpoint = torch.load(model_weights_path, map_location=lambda storage, loc: storage)
            if "state_dict" in checkpoint.keys():
                model.load_state_dict(checkpoint["state_dict"])
            else:
                model.load_state_dict(checkpoint)
        else:
            raise FileNotFoundError("Model weight file not found")
        # Extract the output of the feature extraction layer
        self.feature_extractor = create_feature_extractor(model, feature_nodes)
        # Select the specified layers as the feature extraction layer
        self.feature_extractor_nodes = feature_nodes
        # input normalization
        self.normalize = transforms.Normalize(feature_normalize_mean, feature_normalize_std)
        # Freeze model parameters without derivatives
        for model_parameters in self.feature_extractor.parameters():
            model_parameters.requires_grad = False
        self.feature_extractor.eval()

    def forward(self, sr_tensor: Tensor, gt_tensor: Tensor) -> [Tensor]:
        assert sr_tensor.size() == gt_tensor.size(), "Two tensor must have the same size"
        device = sr_tensor.device

        losses = []
        # input normalization
        sr_tensor = self.normalize(sr_tensor)
        gt_tensor = self.normalize(gt_tensor)

        # Get the output of the feature extraction layer
        sr_feature = self.feature_extractor(sr_tensor)
        gt_feature = self.feature_extractor(gt_tensor)

        # Compute feature loss
        for i in range(len(self.feature_extractor_nodes)):
            losses.append(F_torch.mse_loss(sr_feature[self.feature_extractor_nodes[i]],
                                           gt_feature[self.feature_extractor_nodes[i]]))

        return torch.stack(losses)


def srresnet_x2(**kwargs: Any) -> SRResNet:
    model = SRResNet(upscale=2, **kwargs)

    return model


def srresnet_x4(**kwargs: Any) -> SRResNet:
    model = SRResNet(upscale=4, **kwargs)

    return model


def srresnet_x8(**kwargs: Any) -> SRResNet:
    model = SRResNet(upscale=8, **kwargs)

    return model


def discriminator_for_vgg(**kwargs) -> DiscriminatorForVGG:
    model = DiscriminatorForVGG(**kwargs)

    return model


Writing model_srgan.py


In [ ]:
#Note on Training. Refer to the archived source for the detailed original comment.
!python train_srresnet_exp_scale.py \
    --run_name "lr5e5_wd1e5" \
    --lr 5e-5 \
    --weight_decay 1e-5 \
    --channels 64 \
    --epochs 100 \
    --scale 3 \
    --patience 20 \
    --cache

流式输出内容被截断，只能显示最后 5000 行内容。
Epoch [5/100][ 900/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.003077 (0.002355)
Epoch [5/100][1000/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.002784 (0.002360)
Epoch [5/100][1100/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.002095 (0.002361)
Epoch [5/100][1200/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.003070 (0.002365)
Epoch [5/100][1300/5000]	Time  0.009 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.001608 (0.002362)
Epoch [5/100][1400/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.002509 (0.002365)
Epoch [5/100][1500/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.001733 (0.002364)
Epoch [5/100][1600/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.001619 (0.002356)
Epoch [5/100][1700/5000]	Time  0.009 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.001946 (0.002352)
Epoch [5/100][1800/5000]	Time  0.008 ( 0.009)	Data  0.000 ( 0.000)	Loss 0.003400 (0.002356)
Epoch [5/100][1900/5000]	Time  0.008 ( 0.009)	Data  0

In [ ]:
# Note on Testing. Refer to the archived source for the detailed original comment.
model_paths = [
    "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x3_best_raw.pth",
    "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x3_best_ema.pth"
]
for path in model_paths:
    print(f"\n===== Testing {path} =====")
    !python test_srresnet_exp_scale.py --model_path {path} --scale 3


===== Testing /content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x3_best_raw.pth =====
Set5: PSNR = 34.03 dB, SSIM = 0.9229
Set14: PSNR = 30.12 dB, SSIM = 0.8368
Urban100: PSNR = 27.54 dB, SSIM = 0.8378
BSDS100: PSNR = 28.92 dB, SSIM = 0.7998

===== Testing /content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x3_best_ema.pth =====
Set5: PSNR = 34.09 dB, SSIM = 0.9233
Set14: PSNR = 30.14 dB, SSIM = 0.8373
Urban100: PSNR = 27.59 dB, SSIM = 0.8387
BSDS100: PSNR = 28.94 dB, SSIM = 0.8003


### Training

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.

For ×3, EMA and raw weights produced nearly identical benchmark performance. EMA was selected because it achieved slightly higher DIV2K validation PSNR/SSIM and marginally better SSIM on most benchmark datasets.


### Experiment Notes

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [ ]:
%%writefile train_srgan_exp_scale.py

# train_srgan_exp.py
# Note on Training. Refer to the archived source for the detailed original comment.
# 兼容不同scale

import os
import sys
import time
import argparse
import random
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import MultiStepLR
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.backends import cudnn
from torch.utils.tensorboard import SummaryWriter

sys.path.append('/content/SRGAN-PyTorch')

from sr_dataset_exp_scale import SRDatasetConfig, SRDataset
from model_srgan import SRResNet, discriminator_for_vgg, ContentLoss
from utils import AverageMeter, ProgressMeter
from image_quality_assessment import PSNR, SSIM

# Note on Path. Refer to the archived source for the detailed original comment.
splits_info_multi = {
    'train': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_train_LR_bicubic/X4',
        }
    },

    'valid': {
        'hr_folder': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_HR',
        'lr_folders': {
            '2': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X2',
            '3': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X3',
            '4': '/content/drive/MyDrive/datasets/DIV2K_HRLR/DIV2K_valid_LR_bicubic/X4',
        }
    }
}


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_hr_folder(split_info, scale):
    hr_base = split_info['hr_folder']
    hr_cropped_dir = os.path.join(hr_base, 'HR_cropped', f'X{scale}')
    if os.path.isdir(hr_cropped_dir):
        return hr_cropped_dir
    else:
        return hr_base

def evaluate(eval_model, valid_loader, psnr_model, ssim_model, device):
    """评估模型在验证集上的 PSNR/SSIM"""
    eval_model.eval()
    psnr_vals, ssim_vals = [], []
    with torch.no_grad():
        for lr_img, hr_img in valid_loader:
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            sr = eval_model(lr_img)
            psnr = psnr_model(sr, hr_img)
            ssim = ssim_model(sr, hr_img)
            psnr_vals.append(psnr.item())
            ssim_vals.append(ssim.item())
    avg_psnr = sum(psnr_vals) / len(psnr_vals)
    avg_ssim = sum(ssim_vals) / len(ssim_vals)
    return avg_psnr, avg_ssim

def main():
    parser = argparse.ArgumentParser(description='Train SRGAN')
    # Note on Training, Hyperparameters, Parameters. Refer to the archived source for the detailed original comment.
    parser.add_argument('--lr', type=float, default=1e-4, help='Learning rate for G and D')
    parser.add_argument('--weight_decay', type=float, default=0.0, help='Weight decay')
    parser.add_argument('--channels', type=int, default=64, help='Number of channels in G')
    parser.add_argument('--scale', type=int, default=4, choices=[2, 3, 4], help='Super-resolution scale factor')
    parser.add_argument('--epochs', type=int, default=100, help='Total epochs')
    parser.add_argument('--batch_size', type=int, default=16, help='Batch size')
    parser.add_argument('--crop_size', type=int, default=96, help='Crop size for training')
    parser.add_argument('--patch_num', type=int, default=100, help='Patches per image per epoch')
    parser.add_argument('--milestones', type=int, nargs='+', default=[40, 70, 90], help='LR decay milestones')
    parser.add_argument('--gamma', type=float, default=0.5, help='LR decay factor')

    # Note on Weights. Refer to the archived source for the detailed original comment.
    # GitHub SRGAN x4 config baseline
    parser.add_argument('--pixel_weight', type=float, default=1.0, help='Weight for pixel loss')
    parser.add_argument('--content_weight', type=float, default=1.0, help='Weight for perceptual loss')
    parser.add_argument('--adversarial_weight', type=float, default=1e-3, help='Weight for adversarial loss')

    # Note on Path, Data. Refer to the archived source for the detailed original comment.
    parser.add_argument('--save_dir', type=str, default=None, help='Directory to save models. If omitted, automatically uses output_x{scale}.')
    parser.add_argument('--checkpoint_interval', type=int, default=20, help='Save periodic RAW/EMA checkpoints every N epochs. Use 0 to disable.')
    parser.add_argument('--csv_folder', type=str, default='/content/drive/MyDrive/DIV2K_Results/output_csv', help='CSV folder')
    parser.add_argument('--run_name', type=str, default='srgan', help='Name of this experiment')
    parser.add_argument('--num_workers', type=int, default=2, help='Number of data loading workers')
    parser.add_argument('--cache', action='store_true', help='Enable dataset caching')
    parser.add_argument('--color_augment', action='store_true', help='Enable color augmentation')
    parser.add_argument('--noise_augment', action='store_true', help='Add Gaussian noise to LR')

    # Note on Model, Training. Refer to the archived source for the detailed original comment.
    parser.add_argument(
        '--pretrained_g',
        type=str,
        default=None,
        help='Path to scale-matched pretrained SRResNet generator'
    )
    parser.add_argument('--no_pretrained', action='store_true', help='Train G from scratch')
    args = parser.parse_args()

    # Note on Input. Refer to the archived source for the detailed original comment.
    if args.crop_size != 96:
        raise ValueError(
            "discriminator_for_vgg requires crop_size=96"
        )
    # 固定随机种子
    seed = 0
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = False
    cudnn.deterministic = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scale = args.scale

    # Note on Output. Refer to the archived source for the detailed original comment.
    if args.save_dir is None:
        save_dir = os.path.join(
            "/content/drive/MyDrive/DIV2K_Results",
            f"output_x{scale}"
         )
    else:
        save_dir = args.save_dir

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    print(f"Scale: x{scale}")
    print(f"Results will be saved to: {save_dir}")

    # Note on Path, Data. Refer to the archived source for the detailed original comment.
    train_hr = get_hr_folder(splits_info_multi['train'], scale)
    train_lr = splits_info_multi['train']['lr_folders'][str(scale)]
    valid_hr = get_hr_folder(splits_info_multi['valid'], scale)
    valid_lr = splits_info_multi['valid']['lr_folders'][str(scale)]

    # Note on Training. Refer to the archived source for the detailed original comment.
    train_config = SRDatasetConfig(
        csv_folder=args.csv_folder,
        HR_data_folder=train_hr,
        LR_data_folder=train_lr,
        crop_size=args.crop_size,
        scaling_factor=scale,
        patch_num=args.patch_num,
        cache=args.cache,
        scale=scale,
        color_augment=args.color_augment,
        noise_augment=args.noise_augment
    )
    valid_config = SRDatasetConfig(
        csv_folder=args.csv_folder,
        HR_data_folder=valid_hr,
        LR_data_folder=valid_lr,
        crop_size=-1,
        scaling_factor=scale,
        patch_num=1,
        cache=args.cache,
        scale=scale,
        color_augment=False,
        noise_augment=False
    )

    train_dataset = SRDataset("train", train_config)
    valid_dataset = SRDataset("valid", valid_config)

    g_train = torch.Generator()
    g_train.manual_seed(seed)
    g_valid = torch.Generator()
    g_valid.manual_seed(seed + 1)

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True,
        worker_init_fn=seed_worker,
        generator=g_train
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g_valid
    )

    # Note on Generate. Refer to the archived source for the detailed original comment.
    g_model = SRResNet(channels=args.channels, upscale=scale).to(device)

    # Note on Training, Load, Weights. Refer to the archived source for the detailed original comment.
    if args.no_pretrained:
        print(f"Training x{scale} generator from scratch.")
    else:
        if args.pretrained_g is None:
            raise ValueError(
                f"--pretrained_g must be provided "
                f"for SRGAN x{scale}"
            )

        if not os.path.exists(args.pretrained_g):
            raise FileNotFoundError(
                f"Pretrained G not found: "
                f"{args.pretrained_g}"
            )

        state_dict = torch.load(args.pretrained_g, map_location=device)
        g_model.load_state_dict(state_dict, strict=True)

        print(
            f"Loaded x{scale} pretrained G from:\n"
            f"{args.pretrained_g}"
        )

    # ===== 2. 构建判别器 (D) =====
    d_model = discriminator_for_vgg(in_channels=3, out_channels=1, channels=64).to(device)

    # Note on Loss Function. Refer to the archived source for the detailed original comment.
    pixel_criterion = nn.MSELoss().to(device)
    content_criterion = ContentLoss(
        net_cfg_name="vgg19",
        batch_norm=False,
        num_classes=1000,
        model_weights_path="",   # 自动下载 PyTorch 预训练权重
        feature_nodes=["features.35"],  # VGG19 的 conv5_4
        feature_normalize_mean=[0.485, 0.456, 0.406],
        feature_normalize_std=[0.229, 0.224, 0.225]
    ).to(device)
    adversarial_criterion = nn.BCEWithLogitsLoss().to(device)

    # Note on Optimizer. Refer to the archived source for the detailed original comment.
    g_optimizer = optim.Adam(g_model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    d_optimizer = optim.Adam(d_model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    g_scheduler = MultiStepLR(g_optimizer, milestones=args.milestones, gamma=args.gamma)
    d_scheduler = MultiStepLR(d_optimizer, milestones=args.milestones, gamma=args.gamma)

    scaler = GradScaler()

    # ===== 5. EMA (可选) =====
    ema_decay = 0.999
    ema_g_model = AveragedModel(
        g_model,
        multi_avg_fn=get_ema_multi_avg_fn(ema_decay),
        use_buffers=True
    )
    # Note on Training, Initialize, Weights. Refer to the archived source for the detailed original comment.
    ema_g_model.update_parameters(g_model)

    # Note on Evaluation. Refer to the archived source for the detailed original comment.
    psnr_model = PSNR(crop_border=scale, only_test_y_channel=True).to(device)
    ssim_model = SSIM(crop_border=scale, only_test_y_channel=True).to(device)

    # Note on Save. Refer to the archived source for the detailed original comment.
    run_tag = f"{args.run_name}_x{scale}"
    raw_save_path = os.path.join(save_dir, f'{run_tag}_best_raw.pth')
    ema_save_path = os.path.join(save_dir, f'{run_tag}_best_ema.pth')
    log_path = os.path.join(save_dir, f'{run_tag}_metrics.csv')
    writer = SummaryWriter(os.path.join("samples", "logs", run_tag))

    # Note on Initialize. Refer to the archived source for the detailed original comment.
    with open(log_path, 'w', newline='') as f:
        csv_writer = csv.writer(f)
        csv_writer.writerow([
            'epoch', 'raw_psnr', 'raw_ssim', 'ema_psnr', 'ema_ssim',
            'g_loss', 'd_loss', 'pixel_loss', 'content_loss', 'adversarial_loss',
            'lr'
        ])

    # Note on Training. Refer to the archived source for the detailed original comment.
    best_raw_psnr = 0.0
    best_ema_psnr = 0.0

    # Note on Training, Loop. Refer to the archived source for the detailed original comment.
    for epoch in range(args.epochs):
        g_model.train()
        d_model.train()

        batch_time = AverageMeter('Time', ':6.3f')
        data_time = AverageMeter('Data', ':6.3f')
        g_losses = AverageMeter('G Loss', ':6.6f')
        d_losses = AverageMeter('D Loss', ':6.6f')
        pixel_losses = AverageMeter('Pixel Loss', ':6.6f')
        content_losses = AverageMeter('Content Loss', ':6.6f')
        adv_losses = AverageMeter('Adv Loss', ':6.6f')
        progress = ProgressMeter(len(train_loader), [batch_time, data_time, g_losses, d_losses],
                                 prefix=f"Epoch [{epoch+1}/{args.epochs}]")

        end = time.time()
        for i, (lr_img, hr_img) in enumerate(train_loader):
            lr_img = lr_img.to(device, non_blocking=True)
            hr_img = hr_img.to(device, non_blocking=True)
            data_time.update(time.time() - end)

            # Note on Training, Generate. Refer to the archived source for the detailed original comment.
            # 1) 冻结 D，更新 G
            for p in d_model.parameters():
                p.requires_grad = False

            g_optimizer.zero_grad(set_to_none=True)

            with autocast():
                sr = g_model(lr_img)

                pixel_loss = (
                    args.pixel_weight
                    * pixel_criterion(sr, hr_img)
                )

                content_loss = (
                    args.content_weight
                    * content_criterion(sr, hr_img).mean()
                )

                d_fake_for_g = d_model(sr)

                adversarial_loss = (
                    args.adversarial_weight
                    * adversarial_criterion(
                        d_fake_for_g,
                        torch.ones_like(d_fake_for_g)
                    )
                )

                g_loss = (
                    pixel_loss
                    + content_loss
                    + adversarial_loss
                )

            scaler.scale(g_loss).backward()
            scaler.step(g_optimizer)

            # Note on Training. Refer to the archived source for the detailed original comment.
            # Note on Use. Refer to the archived source for the detailed original comment.
            for p in d_model.parameters():
                p.requires_grad = True

            d_optimizer.zero_grad(set_to_none=True)

            with autocast():
                d_real = d_model(hr_img)

                d_loss_real = adversarial_criterion(
                    d_real,
                    torch.ones_like(d_real)
                )

                d_fake = d_model(sr.detach())

                d_loss_fake = adversarial_criterion(
                    d_fake,
                    torch.zeros_like(d_fake)
                )

                d_loss = d_loss_real + d_loss_fake

            scaler.scale(d_loss).backward()
            scaler.step(d_optimizer)

            # 两个 optimizer 都 step 后再 update scaler
            scaler.update()

            ema_g_model.update_parameters(g_model)

            # 记录指标
            pixel_losses.update(pixel_loss.item(), lr_img.size(0))
            content_losses.update(content_loss.item(), lr_img.size(0))
            adv_losses.update(adversarial_loss.item(), lr_img.size(0))
            g_losses.update(g_loss.item(), lr_img.size(0))
            d_losses.update(d_loss.item(), lr_img.size(0))
            batch_time.update(time.time() - end)
            end = time.time()

            if i % 100 == 0:
                progress.display(i)

        # Note on Validation. Refer to the archived source for the detailed original comment.
        raw_psnr, raw_ssim = evaluate(g_model, valid_loader, psnr_model, ssim_model, device)
        ema_psnr, ema_ssim = evaluate(ema_g_model, valid_loader, psnr_model, ssim_model, device)
        print(f"[Epoch {epoch+1}] RAW: PSNR={raw_psnr:.2f}, SSIM={raw_ssim:.4f} | EMA: PSNR={ema_psnr:.2f}, SSIM={ema_ssim:.4f}")
        print(f"         G Loss={g_losses.avg:.4f} | D Loss={d_losses.avg:.4f}")

        # ---- TensorBoard 日志 ----
        writer.add_scalar('PSNR/Raw', raw_psnr, epoch + 1)
        writer.add_scalar('PSNR/EMA', ema_psnr, epoch + 1)
        writer.add_scalar('Loss/G', g_losses.avg, epoch + 1)
        writer.add_scalar('Loss/D', d_losses.avg, epoch + 1)

        # ---- CSV 日志 ----
        with open(log_path, 'a', newline='') as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow([
                epoch + 1,
                raw_psnr, raw_ssim,
                ema_psnr, ema_ssim,
                g_losses.avg, d_losses.avg,
                pixel_losses.avg,
                content_losses.avg,
                adv_losses.avg,
                g_optimizer.param_groups[0]['lr']
            ])

        # Note on Model, Save. Refer to the archived source for the detailed original comment.
        raw_improved = raw_psnr > best_raw_psnr
        ema_improved = ema_psnr > best_ema_psnr

        if raw_improved:
            best_raw_psnr = raw_psnr
            torch.save(
                g_model.state_dict(),
                raw_save_path
            )
            print(f"  -> Best RAW model saved (PSNR: {best_raw_psnr:.2f})")

        if ema_improved:
            best_ema_psnr = ema_psnr
            torch.save(
                ema_g_model.module.state_dict(),
                ema_save_path
            )
            print(f"  -> Best EMA model saved (PSNR: {best_ema_psnr:.2f})")

        # Note on Save. Refer to the archived source for the detailed original comment.
        # SRGAN 不用 PSNR 做 early stopping: SRGAN 的意义是获得更真实的 perceptual texture
        # SRGAN目标 = perceptual quality

        # Note on Save. Refer to the archived source for the detailed original comment.
        current_epoch = epoch + 1

        should_save_checkpoint = (args.checkpoint_interval > 0 and (
                current_epoch % args.checkpoint_interval == 0
                or current_epoch == args.epochs
                )
            )

        if should_save_checkpoint:
            raw_checkpoint_path = os.path.join(save_dir, f'{run_tag}_epoch_{current_epoch}_raw.pth')
            ema_checkpoint_path = os.path.join(save_dir, f'{run_tag}_epoch_{current_epoch}_ema.pth')
            torch.save(g_model.state_dict(), raw_checkpoint_path)
            torch.save(ema_g_model.module.state_dict(), ema_checkpoint_path)

            print(
                f"  -> Periodic checkpoint saved "
                f"at epoch {current_epoch}"
            )

        g_scheduler.step()
        d_scheduler.step()

    writer.close()
    print("Training finished!")


if __name__ == "__main__":
    main()

Writing train_srgan_exp_scale.py


In [ ]:
!python train_srgan_exp_scale.py \
    --run_name "srgan" \
    --scale 2 \
    --pretrained_g "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x2_best_ema.pth" \
    --lr 1e-4 \
    --weight_decay 0 \
    --pixel_weight 1.0 \
    --content_weight 1.0 \
    --adversarial_weight 1e-3 \
    --epochs 100 \
    --checkpoint_interval 20 \
    --cache

流式输出内容被截断，只能显示最后 5000 行内容。
Epoch [5/100][ 200/5000]	Time  0.018 ( 0.020)	Data  0.000 ( 0.002)	G Loss 0.120224 (0.090524)	D Loss 0.138641 (0.308766)
Epoch [5/100][ 300/5000]	Time  0.018 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.070735 (0.090726)	D Loss 0.247122 (0.345129)
Epoch [5/100][ 400/5000]	Time  0.018 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.082102 (0.091220)	D Loss 0.411132 (0.319148)
Epoch [5/100][ 500/5000]	Time  0.018 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.081545 (0.091453)	D Loss 0.343577 (0.333811)
Epoch [5/100][ 600/5000]	Time  0.018 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.062384 (0.091629)	D Loss 1.521268 (0.354106)
Epoch [5/100][ 700/5000]	Time  0.018 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.087617 (0.091361)	D Loss 0.107998 (0.336859)
Epoch [5/100][ 800/5000]	Time  0.019 ( 0.019)	Data  0.000 ( 0.001)	G Loss 0.089950 (0.091642)	D Loss 0.043695 (0.336219)
Epoch [5/100][ 900/5000]	Time  0.019 ( 0.019)	Data  0.000 ( 0.000)	G Loss 0.061715 (0.091480)	D Loss 0.102557 (0.329042)
Epoch

In [ ]:
!python train_srgan_exp_scale.py \
    --run_name "srgan" \
    --scale 3 \
    --pretrained_g "/content/drive/MyDrive/DIV2K_Results/lr5e5_wd1e5_x3_best_ema.pth" \
    --lr 1e-4 \
    --weight_decay 0 \
    --pixel_weight 1.0 \
    --content_weight 1.0 \
    --adversarial_weight 1e-3 \
    --epochs 100 \
    --checkpoint_interval 20 \
    --cache

流式输出内容被截断，只能显示最后 5000 行内容。
Epoch [5/100][1700/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.258391 (0.194567)	D Loss 0.009346 (0.122352)
Epoch [5/100][1800/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.217188 (0.194374)	D Loss 0.016317 (0.122622)
Epoch [5/100][1900/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.175836 (0.193758)	D Loss 0.037167 (0.122808)
Epoch [5/100][2000/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.206802 (0.194228)	D Loss 0.000481 (0.121339)
Epoch [5/100][2100/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.179355 (0.194353)	D Loss 0.002184 (0.123317)
Epoch [5/100][2200/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.206638 (0.194532)	D Loss 0.016951 (0.126644)
Epoch [5/100][2300/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.208278 (0.194728)	D Loss 0.006042 (0.127667)
Epoch [5/100][2400/5000]	Time  0.017 ( 0.017)	Data  0.000 ( 0.000)	G Loss 0.113715 (0.194929)	D Loss 0.005385 (0.124787)
Epoch

### Visualization

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [51]:
import os, sys
sys.path.append('/content/SRGAN-PyTorch')
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from sr_dataset_exp_scale import SRDatasetConfig, SRDataset
from model_srgan import SRResNet
from image_quality_assessment import PSNR, SSIM

def tensor_to_numpy(t):
    """Tensor [1,C,H,W] 或 [C,H,W] → numpy [H,W,C]"""
    if t.dim() == 4:
        t = t.squeeze(0)
    t = t.detach().cpu().clamp(0, 1).numpy()
    return np.transpose(t, (1, 2, 0))

def visualize_compare(scale=2, benchmark="Set5", run_tag="srgan_x2"):
    srresnet_path = (
        f"/content/drive/MyDrive/DIV2K_Results/"
        f"lr5e5_wd1e5_x{scale}_best_ema.pth"
    )

    output_dir = (f"/content/drive/MyDrive/DIV2K_Results/output_x{scale}")

    # Note on Path. Refer to the archived source for the detailed original comment.
    split_info = splits_info_multi[benchmark]
    hr_folder = get_hr_folder(split_info, scale)
    lr_folder = split_info['lr_folders'][str(scale)]

    print("=" * 70)
    print(f"Benchmark : {benchmark}")
    print(f"Scale     : x{scale}")
    print(f"HR folder : {hr_folder}")
    print(f"LR folder : {lr_folder}")
    print("=" * 70)

    # Note on Path, Evaluation. Refer to the archived source for the detailed original comment.
    checkpoint_paths = {}
    for ep in [20, 40, 60, 80, 100]:
        checkpoint_paths[ep] = os.path.join(
            output_dir,
            f"{run_tag}_epoch_{ep}_ema.pth"
        )

    checkpoint_paths["best_ema"] = os.path.join(
        output_dir,
        f"{run_tag}_best_ema.pth"
    )

    checkpoint_paths["best_raw"] = os.path.join(
        output_dir,
        f"{run_tag}_best_raw.pth"
    )

    # Note on Dataset, Data. Refer to the archived source for the detailed original comment.
    config = SRDatasetConfig(
        csv_folder="/content/drive/MyDrive/DIV2K_Results/output_csv",
        HR_data_folder=hr_folder,
        LR_data_folder=lr_folder,
        crop_size=-1,
        scaling_factor=scale,
        patch_num=1,
        cache=False,
        scale=scale,
        color_augment=False,
        noise_augment=False
    )
    dataset = SRDataset(benchmark, config)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    # Note on Evaluation. Refer to the archived source for the detailed original comment.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    psnr_model = PSNR(crop_border=scale, only_test_y_channel=True).to(device)
    ssim_model = SSIM(crop_border=scale, only_test_y_channel=True).to(device)

    # Note on Model, Evaluation. Refer to the archived source for the detailed original comment.
    results = {}

    # 1. SRResNet baseline
    srresnet = SRResNet(channels=64, upscale=scale).to(device)
    srresnet.load_state_dict(torch.load(srresnet_path, map_location=device))
    srresnet.eval()
    srresnet_psnr, srresnet_ssim = evaluate_model(srresnet, loader, psnr_model, ssim_model, device)
    results['SRResNet'] = {'psnr': srresnet_psnr, 'ssim': srresnet_ssim}
    print(f"SRResNet Baseline: PSNR={srresnet_psnr:.2f} dB, SSIM={srresnet_ssim:.4f}")

    # 2. SRGAN checkpoints
    for ep_label, ckpt_path in checkpoint_paths.items():
        if not os.path.exists(ckpt_path):
            print(f"⚠️ Checkpoint not found: {ckpt_path}")
            continue
        model = SRResNet(channels=64, upscale=scale).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()
        avg_psnr, avg_ssim = evaluate_model(model, loader, psnr_model, ssim_model, device)
        results[ep_label] = {'psnr': avg_psnr, 'ssim': avg_ssim, 'model': model}
        print(f"{ep_label:20}: PSNR={avg_psnr:.2f} dB, SSIM={avg_ssim:.4f}")

    # ========== 选出最佳 GAN ==========
    gan_keys = [k for k in results.keys() if k != 'SRResNet']
    if gan_keys:
        best_gan_label = max(gan_keys, key=lambda k: results[k]['psnr'])
        print(f"\nBest GAN checkpoint: {best_gan_label} (PSNR={results[best_gan_label]['psnr']:.2f})")
    else:
        raise RuntimeError("No valid SRGAN checkpoint found!")

    # Note on Visualization. Refer to the archived source for the detailed original comment.
    num_display = min(5, len(dataset))
    indices = np.random.choice(len(dataset), num_display, replace=False)

    fig, axes = plt.subplots(num_display, 5, figsize=(20, 4 * num_display))
    for i, idx in enumerate(indices):
        lr, hr = dataset[idx]
        lr = lr.unsqueeze(0).to(device)
        hr = hr.unsqueeze(0).to(device)

        lr_display = F.interpolate(lr, size=hr.shape[-2:], mode='nearest')
        bicubic = F.interpolate(lr, scale_factor=scale, mode='bicubic', align_corners=False)

        with torch.no_grad():
            sr_srresnet = srresnet(lr)
            sr_bestgan = results[best_gan_label]['model'](lr)

        axes[i, 0].imshow(tensor_to_numpy(lr_display))
        axes[i, 0].set_title("LR (Input)", fontsize=10)

        axes[i, 1].imshow(tensor_to_numpy(bicubic))
        axes[i, 1].set_title("Bicubic (Baseline)", fontsize=10)

        axes[i, 2].imshow(tensor_to_numpy(sr_srresnet))
        axes[i, 2].set_title(f"SRResNet (Best)\nPSNR: {results['SRResNet']['psnr']:.2f}", fontsize=9)

        axes[i, 3].imshow(tensor_to_numpy(sr_bestgan))
        axes[i, 3].set_title(f"SRGAN ({best_gan_label})\nPSNR: {results[best_gan_label]['psnr']:.2f}", fontsize=9)

        axes[i, 4].imshow(tensor_to_numpy(hr))
        axes[i, 4].set_title("HR (Ground Truth)", fontsize=10)

        for ax in axes[i]:
            ax.axis('off')

    fig.suptitle(f"×{scale} Super-Resolution Visual Comparison", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_compare(scale=2, benchmark="Set5", run_tag="srgan_x2")

Benchmark : Set5
Scale     : x2
HR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR/HR_cropped/X2
LR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X2
SRResNet Baseline: PSNR=37.80 dB, SSIM=0.9587
                  20: PSNR=35.40 dB, SSIM=0.9398
                  40: PSNR=35.46 dB, SSIM=0.9387
                  60: PSNR=35.47 dB, SSIM=0.9383
                  80: PSNR=35.59 dB, SSIM=0.9379
                 100: PSNR=35.43 dB, SSIM=0.9366
best_ema            : PSNR=35.99 dB, SSIM=0.9496
best_raw            : PSNR=35.80 dB, SSIM=0.9375

Best GAN checkpoint: best_ema (PSNR=35.99)


<Figure size 2000x2000 with 25 Axes>

### Experiment Notes

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [55]:
visualize_compare(scale=3, benchmark="Set5", run_tag="srgan_x3")

Benchmark : Set5
Scale     : x3
HR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR/HR_cropped/X3
LR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X3
SRResNet Baseline: PSNR=34.09 dB, SSIM=0.9233
                  20: PSNR=31.76 dB, SSIM=0.8960
                  40: PSNR=32.11 dB, SSIM=0.8966
                  60: PSNR=32.27 dB, SSIM=0.8977
                  80: PSNR=32.21 dB, SSIM=0.8955
                 100: PSNR=32.35 dB, SSIM=0.8952
best_ema            : PSNR=32.35 dB, SSIM=0.8952
best_raw            : PSNR=32.34 dB, SSIM=0.8924

Best GAN checkpoint: 100 (PSNR=32.35)


<Figure size 2000x2000 with 25 Axes>

### Experiment Notes

This cell documents the image super-resolution experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [57]:
visualize_compare(scale=4, benchmark="Set5", run_tag="srgan_baseline_100")

Benchmark : Set5
Scale     : x4
HR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/HR
LR folder : /content/drive/MyDrive/datasets/DIV2K_HRLR/benchmark/Set5/LR_bicubic/X4
SRResNet Baseline: PSNR=31.88 dB, SSIM=0.8899
                  20: PSNR=29.68 dB, SSIM=0.8501
                  40: PSNR=29.82 dB, SSIM=0.8497
                  60: PSNR=30.04 dB, SSIM=0.8519
                  80: PSNR=30.08 dB, SSIM=0.8513
                 100: PSNR=29.95 dB, SSIM=0.8477
best_ema            : PSNR=30.06 dB, SSIM=0.8499
best_raw            : PSNR=30.13 dB, SSIM=0.8522

Best GAN checkpoint: best_raw (PSNR=30.13)


<Figure size 2000x2000 with 25 Axes>